In [1]:
from pathlib import Path
import pandas as pd

from trino_stack.lakehouse import Lakehouse
from analysis import analyse_workload_set
from analysis import capture_query_plans_for_set


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

WORKLOAD_ROOT = Path("/mnt/primary/Main/Workloads")
RESULTS_DIR = Path("results")

WORKLOAD_NAMES = ["gpt_generated_1000_tpcds_tokens",
     "sqlstorm_1000_tokens_tpcds", 
     "sqlbarber_1000_tokens_tpcds", 
     "sql_factory_1000_tokens_tpcds",
     "e2etune_1000_tokens_tpcds", 
     "bootstrapping_lcm_1000_tokens_tpcds"
     ]

WORKLOAD_NAMES = [
                  "tpcds_1000",
               ]

CATALOG = "iceberg"
SCHEMA = "tpcds"


lh = Lakehouse.from_release(
    instance_name="lakehouse-g",
    namespace="pgr24james",
    sync_schemas=True,
    verbose=False,
)

Checking Trino health ...


In [2]:

overview_df, failures_df = analyse_workload_set(
    WORKLOAD_NAMES,
    workload_root=WORKLOAD_ROOT,
    lh=lh,
    catalog=CATALOG,
    schema=SCHEMA,
)

overview_df

overview_df.to_csv(
    RESULTS_DIR / "tpcds_1000.csv",
    index=False,
)

if not failures_df.empty:
    print("\n=== Failures ===")
    display(failures_df)


=== Analysing workload: tpcds_1000 ===
  workload  sql_queries  planned_queries  table_coverage_pct  column_coverage_pct  join_edge_coverage_pct  table_usage_entropy  column_usage_entropy  join_edge_usage_entropy  unique_table_set_ratio  unique_plan_graphs  plan_uniqueness_ratio  operator_types_observed  operator_type_entropy  unique_operator_instances  operator_ngram_entropy  mean_nn_plan_distance  min_nn_plan_distance  plan_graph_vendi_score
tpcds_1000         1000             1000               100.0                61.18                   68.29               0.8435                0.8732                   0.8214                   0.065                  87                  0.087                       28                 0.6858                       4718                  0.7243                    0.0                   0.0                 22.7182


In [ ]:
# ---------------------------------------------------------------------
# Generate query plans (EXPLAIN JSON + DAGs) for each workload.
# ---------------------------------------------------------------------

plans_summary_df, plans_failures_df = capture_query_plans_for_set(
    WORKLOAD_NAMES,
    workload_root=WORKLOAD_ROOT,
    lh=lh,
    catalog=CATALOG,
    schema=SCHEMA,
    force_recompute=False,
    plan_workers: int = 8,
)

plans_summary_df

if not plans_failures_df.empty:
    print("\n=== Plan capture failures ===")
    display(plans_failures_df)


=== Capturing plans: gpt_generated_1000_tpcds_tokens ===
gpt_generated_1000_tpcds_tokens: capturing plans for 1000 queries ...
